# MinbarAI — TranslateGemma QLoRA fine-tune (Kaggle)

**Setup:** `Settings > Accelerator > GPU T4 x2`, `Internet > On`, and add a Kaggle secret named `HF_TOKEN` (Add-ons > Secrets) with a HuggingFace WRITE token. Accept the `google/translategemma-12b-it` license on HF once.

Set `DRY_RUN = True` for the 4B pipeline-validation pass (single T4, ~3-4 h), `False` for the real 12B run (~8-15 h; checkpoints push to HF continuously, rerun the notebook to resume).

In [ ]:
DRY_RUN = True   # <-- flip to False for the 12B mission run
HF_USER = "CHANGE_ME"  # your HF username

MODEL = "google/translategemma-4b-it" if DRY_RUN else "google/translategemma-12b-it"
REPO = f"{HF_USER}/translategemma-{'4b' if DRY_RUN else '12b'}-khutbah-lora"

In [ ]:
# 1. Code + deps + HF auth
!git clone -b cloud-pipeline https://github.com/Yacine-DH/MinbarAI.git
%cd MinbarAI
!pip install -q -U transformers peft bitsandbytes datasets accelerate rapidfuzz sentencepiece

from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!huggingface-cli login --token $HF_TOKEN

In [ ]:
# 2. Build the dataset (~24k pairs, eval cases excluded)
!python training/build_dataset.py

In [ ]:
# 3. Train (resumable — checkpoints push to the Hub every save)
!python training/finetune_qlora.py --model $MODEL --hub-repo $REPO

In [ ]:
# 4. Merge adapter into the base model and push the merged model
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merged_repo = REPO.replace("-lora", "")
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cpu")
model = PeftModel.from_pretrained(base, REPO)
model = model.merge_and_unload()
tok = AutoTokenizer.from_pretrained(MODEL)
model.push_to_hub(merged_repo)
tok.push_to_hub(merged_repo)
print("merged ->", merged_repo)

In [ ]:
# 5. GGUF Q4_K_M + push (for Ollama serving on Modal)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!huggingface-cli download {merged_repo} --local-dir merged_hf
!python llama.cpp/convert_hf_to_gguf.py merged_hf --outfile model-f16.gguf --outtype f16
!cmake -B llama.cpp/build llama.cpp -DGGML_CUDA=OFF && cmake --build llama.cpp/build --target llama-quantize -j4
!llama.cpp/build/bin/llama-quantize model-f16.gguf model-Q4_K_M.gguf Q4_K_M
!huggingface-cli upload {merged_repo} model-Q4_K_M.gguf model-Q4_K_M.gguf
print("GGUF pushed. Serve on Modal by setting MODEL_ID to an Ollama model created FROM this GGUF (see KAGGLE_MISSION.md step 5).")